# Pet Recognition Training Notebook

This notebook helps you train a custom pet recognizer (`pets.pkl`) to identify individual pets by name.

**Workflow:**
1. Extract pet crops from YOLOv12 detections
2. Manually label 50-100 crops per pet
3. Train a fastai ResNet classifier
4. Export the model as `pets.pkl`
5. Use in the main pipeline for automatic pet recognition

**Requirements:**
```bash
pip install fastai torch torchvision
```

In [ ]:
# Imports
import sys
from pathlib import Path
import shutil
from collections import Counter

# Add backend to path
sys.path.insert(0, str(Path.cwd().parent / 'backend'))

from database import session_scope
from models import MediaFile, PetEmbedding
from config import Config
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

cfg = Config()
print(f"Database: {cfg.DB_PATH}")
print(f"Organized folder: {cfg.ORGANIZED_DIR}")

AttributeError: 'Config' object has no attribute 'DATABASE_PATH'

## Step 1: Extract Unlabeled Pet Crops

Run the pipeline first to detect pets with YOLOv12, then extract crops here for labeling.

In [ ]:
# Create directory structure for labeling
LABELING_DIR = Path.cwd() / 'pet_crops_unlabeled'
LABELED_DIR = Path.cwd() / 'pet_crops_labeled'

LABELING_DIR.mkdir(exist_ok=True)
LABELED_DIR.mkdir(exist_ok=True)

# Extract pet crops from database
with session_scope() as session:
    pets = session.query(PetEmbedding).join(MediaFile).filter(
        PetEmbedding.cluster_id.is_(None)  # Only unlabeled pets
    ).limit(500).all()  # Start with 500 samples
    
    print(f"Found {len(pets)} unlabeled pet detections")
    
    for i, pet in enumerate(pets):
        img_path = Path(pet.media.path)
        if not img_path.exists():
            continue
        
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        bbox = pet.bbox
        x, y, w, h = bbox['x'], bbox['y'], bbox['w'], bbox['h']
        crop = img[y:y+h, x:x+w]
        
        if crop.size == 0:
            continue
        
        # Save crop
        crop_path = LABELING_DIR / f"pet_{i:04d}_{pet.species}.jpg"
        cv2.imwrite(str(crop_path), crop)

print(f"\nExtracted crops saved to: {LABELING_DIR}")
print(f"\nNext steps:")
print(f"1. Open {LABELING_DIR} in File Explorer")
print(f"2. Create folders in {LABELED_DIR} for each pet (e.g., 'Fido', 'Sparky')")
print(f"3. Move 50-100 crops of each pet into their folder")
print(f"4. Continue to next cell when done")

## Step 2: Verify Labeled Data

Check the labeled dataset structure.

In [ ]:
# Count samples per pet
pet_counts = {}
for pet_folder in LABELED_DIR.iterdir():
    if pet_folder.is_dir():
        count = len(list(pet_folder.glob('*.jpg')))
        pet_counts[pet_folder.name] = count

print("Labeled dataset:")
for pet_name, count in sorted(pet_counts.items()):
    print(f"  {pet_name}: {count} images")

total = sum(pet_counts.values())
print(f"\nTotal: {total} labeled images across {len(pet_counts)} pets")

if total < 100:
    print("\n⚠️  Warning: Less than 100 total images. Recommend at least 50-100 per pet.")
elif total < 200:
    print("\n✓ Good dataset size for initial training")
else:
    print("\n✓ Excellent dataset size!")

## Step 3: Show Sample Images

Visualize a few samples from each pet to verify labeling.

In [ ]:
# Show 3 random samples from each pet
import random

for pet_folder in sorted(LABELED_DIR.iterdir()):
    if not pet_folder.is_dir():
        continue
    
    images = list(pet_folder.glob('*.jpg'))
    if not images:
        continue
    
    samples = random.sample(images, min(3, len(images)))
    
    print(f"\n{pet_folder.name}:")
    fig, axes = plt.subplots(1, len(samples), figsize=(12, 4))
    if len(samples) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(img_path.stem)
    
    plt.tight_layout()
    plt.show()

## Step 4: Train fastai Pet Recognizer

Train a ResNet classifier on the labeled data.

In [ ]:
from fastai.vision.all import *

# Create DataLoaders
dls = ImageDataLoaders.from_folder(
    LABELED_DIR,
    valid_pct=0.2,  # 20% validation set
    seed=42,
    item_tfms=Resize(224),  # Resize all images to 224x224
    batch_tfms=aug_transforms(size=224, min_scale=0.75)  # Data augmentation
)

# Show a batch
dls.show_batch(max_n=9, figsize=(10, 10))

In [ ]:
# Create and train the model
learn = vision_learner(
    dls,
    resnet34,  # ResNet34 - good balance of speed and accuracy
    metrics=[error_rate, accuracy]
)

# Fine-tune for 5 epochs
learn.fine_tune(5)

## Step 5: Evaluate Model Performance

In [ ]:
# Show confusion matrix
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(8, 8))

In [ ]:
# Show top losses (misclassified images)
interp.plot_top_losses(9, figsize=(15, 10))

## Step 6: Export Model

Save the trained model as `pets.pkl` for use in the pipeline.

In [ ]:
# Export model
MODEL_PATH = Path.cwd().parent / 'backend' / 'models' / 'pets.pkl'
MODEL_PATH.parent.mkdir(exist_ok=True)

learn.export(MODEL_PATH)

print(f"✓ Model exported to: {MODEL_PATH}")
print(f"\nPet classes:")
for i, pet_name in enumerate(learn.dls.vocab):
    print(f"  {i}: {pet_name}")

print(f"\n✅ Pet recognizer ready!")
print(f"\nNext steps:")
print(f"1. Run the pipeline: python run_pipeline.py")
print(f"2. The pipeline will now automatically recognize these pets")
print(f"3. Browse pets at: http://localhost:5052/api/pets/")

## Step 7: Test the Model

Test the model on a few sample images.

In [ ]:
# Load the exported model
learn_inf = load_learner(MODEL_PATH)

# Test on a few random images
test_images = random.sample(list(LABELED_DIR.rglob('*.jpg')), min(6, len(list(LABELED_DIR.rglob('*.jpg')))))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, test_images):
    img = Image.open(img_path)
    pred, pred_idx, probs = learn_inf.predict(img)
    
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"True: {img_path.parent.name}\nPred: {pred} ({probs[pred_idx]:.2%})")

plt.tight_layout()
plt.show()